# 원본 Kaggle 탐색 실험 (출력 제거본)

jaewon1007의 LG_Detection 원본 코드 흐름을 보존한 자료입니다. 실행 순서 의존성, 중복 함수와 생성 실험을 포함합니다. 신규 실행에는 `01_rdd_walkthrough.ipynb`를 권장합니다.

참고: [Choi et al., CVPR 2026, RDD](https://www.lgresearch.ai/publication/view?seq=166).
추가 학습은 없지만 사전학습 신경망을 사용합니다. 공식 구현이나 전체 논문 재현이 아닙니다.

공개본에서는 출력/이미지, 실행 메타데이터, 개인 파일 경로를 제거했고 pip 설치 목록을 명시했습니다. 두 로컬 예시 셀은 경로를 지정한 후 직접 실행하도록 자동 호출을 주석 처리했습니다. 기존 수치는 `docs/experiments.md`에 별도 보존했습니다.


## 모델 로딩

In [ ]:
%pip install -q torch torchvision "diffusers<1" lpips accelerate "transformers<5" requests numpy tqdm Pillow

import torch
import torchvision.transforms as T
from diffusers import AutoencoderKL, UNet2DConditionModel
import lpips

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 3. 모델 불러오기 (논문에서 검증한 Stable Diffusion v1.4 기준)
model_id = "CompVis/stable-diffusion-v1-4"

print("Loading VAE (Autoencoder)...")
vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae")
vae.to(device).eval()        
vae.requires_grad_(False)     

print("Loading UNet (Noise Predictor)...")
unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet")
unet.to(device).eval()
unet.requires_grad_(False)

print("Loading LPIPS...")
loss_fn_vgg = lpips.LPIPS(net='vgg').to(device).eval()
loss_fn_vgg.requires_grad_(False)

print("Ready")

## 이미지 점수

In [ ]:
import torch.nn.functional as F

# 논문에서 제시한 하이퍼파라미터 세팅
LAMBDA_R = 0.5  
blur_transform = T.GaussianBlur(kernel_size=3, sigma=0.8)

def get_f_AE(x):
    """
    이미지 x와 이를 VAE로 압축-복원한 이미지 간의 LPIPS 거리를 계산합니다.
    입력 x는 [-1, 1] 범위로 정규화된 (B, C, H, W) 텐서라고 가정합니다.
    """
    with torch.no_grad():
        # 1. 인코더를 통해 잠재 벡터(Latent)로 압축
        latent = vae.encode(x).latent_dist.sample()
        # Stable Diffusion VAE의 기본 스케일링 팩터 적용
        latent = latent * vae.config.scaling_factor 
        
        # 2. 디코더를 통해 이미지로 다시 복원
        recon = vae.decode(latent / vae.config.scaling_factor).sample
        
        # 3. LPIPS 오차 계산 (평균값 반환)
        loss = loss_fn_vgg(x, recon)
        
    return loss.mean()

def get_S_image(x):
    """
    논문의 수식에 따라 이미지 수준의 편향이 제거된 최종 점수를 계산합니다.
    """
    # 1. 원본 이미지의 오차
    f_x = get_f_AE(x)
    
    # 2. 저주파 필터(가우시안 블러) 적용 후 오차
    F_x = blur_transform(x)
    f_F_x = get_f_AE(F_x)
    
    # 3. 90도 회전 적용 후 오차 (공간 차원인 dim 2, 3을 기준으로 회전)
    R_x = torch.rot90(x, k=1, dims=[2, 3])
    f_R_x = get_f_AE(R_x)
    
    # 4. 회전과 블러를 동시에 적용 후 오차
    F_R_x = blur_transform(R_x)
    f_F_R_x = get_f_AE(F_R_x)
    
    # 5. 최종 S_image 수식 적용
    S_image = f_x - f_F_x - (LAMBDA_R * f_R_x) + (LAMBDA_R * f_F_R_x)
    
    return S_image.item()

print("✅ S_image 계산 함수 정의 완료!")

## Picsum 단일 예시

In [ ]:
import requests
from PIL import Image
from io import BytesIO
import torchvision.transforms as T

url = "https://picsum.photos/512"
response = requests.get(url)
response.raise_for_status() # 통신에 문제가 있으면 여기서 즉시 에러를 발생시킵니다.

raw_image = Image.open(BytesIO(response.content)).convert("RGB")

# 원본 이미지 확인 (매번 실행할 때마다 다른 풍경/사물 사진이 나옵니다)
display(raw_image.resize((256, 256))) 

# 2. 이미지 전처리 (Stable Diffusion 입력 형식: 512x512, [-1, 1] 범위 정규화)
preprocess = T.Compose([
    T.Resize((512, 512)),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

img_tensor = preprocess(raw_image).unsqueeze(0).to(device)

# 3. S_image 점수 계산
print("🔄 S_image 점수를 계산 중입니다. 잠시만 기다려주세요...")
score = get_S_image(img_tensor)

print(f"✅ 테스트 이미지의 S_image 점수: {score:.5f}")

## 잠재 점수와 RDD

In [ ]:
import torch.nn.functional as F
from diffusers import DDPMScheduler
from transformers import CLIPTextModel, CLIPTokenizer

print("Loading Tokenizer, Text Encoder & Scheduler...")
tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder").to(device).eval()
text_encoder.requires_grad_(False)
scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")

def get_f_latent(z_0, t_ratio=0.05, n=16):
    """
    잠재 벡터 z_0에 노이즈를 추가하고, UNet이 이를 얼마나 잘 예측하는지(오차)를 계산합니다.
    - t_ratio: 0.05 (전체 확산 스텝의 5% 시점)
    - n: 16 (노이즈 샘플링 횟수)
    """
    # 빈 텍스트(Null-text) 프롬프트 생성
    text_input = tokenizer([""], padding="max_length", max_length=tokenizer.model_max_length, return_tensors="pt")
    with torch.no_grad():
        null_prompt_embeds = text_encoder(text_input.input_ids.to(device))[0]
    
    # 배치 사이즈(n)에 맞게 프롬프트 복제
    null_prompt_embeds = null_prompt_embeds.repeat(n, 1, 1)
    
    # 타임스텝 설정 (예: 1000 * 0.05 = 50)
    num_train_timesteps = scheduler.config.num_train_timesteps
    t = int(num_train_timesteps * t_ratio)
    t_tensor = torch.tensor([t] * n, device=device)
    
    # 잠재 벡터 복제 및 노이즈 생성
    z_0_batched = z_0.repeat(n, 1, 1, 1)
    noise = torch.randn_like(z_0_batched)
    z_t = scheduler.add_noise(z_0_batched, noise, t_tensor)
    
    # UNet을 통해 더해진 노이즈 예측
    with torch.no_grad():
        noise_pred = unet(z_t, t_tensor, encoder_hidden_states=null_prompt_embeds).sample
        
    # 실제 더한 노이즈와 예측된 노이즈의 MSE(평균제곱오차) 반환
    loss = F.mse_loss(noise_pred, noise)
    return loss.item()

def get_S_latent(x, t_ratio=0.05, n=16):
    """
    논문의 수식에 따라 잠재 공간에서 편향이 제거된 최종 점수를 계산합니다.
    """
    with torch.no_grad():
        # 이미지를 잠재 공간으로 압축 (z_0)
        z_0 = vae.encode(x).latent_dist.sample()
        z_0 = z_0 * vae.config.scaling_factor
        
    # 1. 원본 잠재 벡터 오차
    f_z_0 = get_f_latent(z_0, t_ratio, n)
    
    # 2. 90도 회전된 잠재 벡터 오차
    R_z_0 = torch.rot90(z_0, k=1, dims=[2, 3])
    f_R_z_0 = get_f_latent(R_z_0, t_ratio, n)
    
    # 3. 회전된 오차에서 원본 오차를 빼서 편향 제거
    S_latent = f_R_z_0 - f_z_0
    
    return S_latent

def get_RDD_score(x):
    """
    이미지 점수와 잠재 점수를 통합하여 최종 S_RDD 점수를 반환합니다.
    """
    score_img = get_S_image(x)
    score_lat = get_S_latent(x)
    
    # 논문 수식: S_image * (S_latent)^2
    final_score = score_img * (score_lat ** 2)
    return final_score, score_img, score_lat

# 앞서 사용한 이미지 텐서를 그대로 적용하여 최종 점수 확인
print("🔄 최종 RDD 점수를 계산 중입니다. GPU가 열심히 일하는 중...")
score_rdd, s_img, s_lat = get_RDD_score(img_tensor)

print("-" * 40)
print(f"📊 S_image 점수  : {s_img:.5f}")
print(f"📊 S_latent 점수 : {s_lat:.5f}")
print(f"🏆 최종 S_RDD 점수: {score_rdd:.7f}")
print("-" * 40)

## SD 이미지 생성 예시

In [ ]:
from diffusers import StableDiffusionPipeline

# 1. 이미 메모리에 있는 모델들을 조립하여 파이프라인 생성 (추가 다운로드 없음)
pipe = StableDiffusionPipeline(
    vae=vae,
    text_encoder=text_encoder,
    tokenizer=tokenizer,
    unet=unet,
    scheduler=scheduler,
    safety_checker=None,
    feature_extractor=None
).to(device)

# 메모리 절약을 위해 그라디언트 비활성화
pipe.set_progress_bar_config(disable=True)

# 2. AI 이미지 직접 생성
print("🎨 Stable Diffusion이 가짜 이미지를 생성하고 있습니다...")
prompt = "A high quality photo of a cute dog playing in the park"

# 결과 재현성을 위해 시드(Seed) 고정
generator = torch.Generator(device).manual_seed(42)
ai_image = pipe(prompt, num_inference_steps=30, generator=generator).images[0]

# 생성된 AI 이미지 확인
display(ai_image.resize((256, 256)))

# 3. 이미지 전처리
ai_img_tensor = preprocess(ai_image).unsqueeze(0).to(device)

# 4. RDD 점수 계산 및 비교
print("🔄 AI 이미지의 RDD 점수를 계산 중입니다...")
ai_score_rdd, ai_s_img, ai_s_lat = get_RDD_score(ai_img_tensor)

print("\n" + "=" * 45)
print(" 🏁 [최종 탐지 결과 비교]")
print("=" * 45)
print(f"✅ [Real 사진] 이전 S_RDD 점수 : {score_rdd:.7f}")
print("-" * 45)
print(f"🤖 [AI  사진] S_image 점수     : {ai_s_img:.5f}")
print(f"🤖 [AI  사진] S_latent 점수    : {ai_s_lat:.5f}")
print(f"🚨 [AI  사진] 최종 S_RDD 점수  : {ai_score_rdd:.7f}")
print("=" * 45)

## 로컬 Real 예시 (경로 지정 필요)

In [ ]:
import os
from PIL import Image

def classify_my_image(image_path, threshold=0.0001):
    """
    업로드한 로컬 이미지를 불러와 RDD 점수를 계산하고 AI 생성 여부를 판별합니다.
    """
    # 파일 존재 여부 확인
    if not os.path.exists(image_path):
        print(f"❌ 파일을 찾을 수 없습니다. 경로를 다시 확인해주세요: {image_path}")
        return
        
    print(f"🔍 이미지 분석을 시작합니다...")
    
    # 1. 이미지 로드 및 RGB 변환 (PNG 등에서 알파 채널 제거)
    raw_image = Image.open(image_path).convert("RGB")
    
    # 분석할 원본 이미지 화면에 출력
    display(raw_image.resize((256, 256)))
    
    # 2. 전처리 (이전 셀에서 정의한 preprocess 사용)
    img_tensor = preprocess(raw_image).unsqueeze(0).to(device)
    
    # 3. RDD 점수 계산
    score_rdd, s_img, s_lat = get_RDD_score(img_tensor)
    
    # 4. 결과 판별 및 리포트 출력
    print("\n" + "=" * 50)
    print(" 📊 [분석 결과 리포트]")
    print("=" * 50)
    print(f"🔹 S_image 점수 : {s_img:.5f}")
    print(f"🔹 S_latent 점수: {s_lat:.5f}")
    print(f"🏆 최종 RDD 점수: {score_rdd:.7f} (기준값: {threshold})")
    print("-" * 50)
    
    # 기준값(Threshold)과 비교하여 최종 판정
    if score_rdd > threshold:
        print("📸 최종 판정: 이 사진은 [원본(Real) 사진]일 확률이 높습니다!")
    else:
        print("🤖 최종 판정: 이 사진은 [AI가 생성한 이미지]일 확률이 높습니다!")
    print("=" * 50)

# =====================================================================
# 🚀 실행 영역 (여기에 복사한 파일 경로를 붙여넣으세요!)
# =====================================================================
my_file_path = "path/to/image.jpg"

# 만약 캐글 우측 패널 업로드가 번거롭다면, 
# 코드 셀에서 직접 다운로드 받은 임시 파일 경로를 사용해도 됩니다.
# 예: my_file_path = "/kaggle/working/test_image.jpg"

# classify_my_image(my_file_path, threshold=0.0001)


## 로컬 AI 예시 (경로 지정 필요)

In [ ]:
import os
from PIL import Image

def classify_my_image(image_path, threshold=0.0001):
    """
    업로드한 로컬 이미지를 불러와 RDD 점수를 계산하고 AI 생성 여부를 판별합니다.
    """
    # 파일 존재 여부 확인
    if not os.path.exists(image_path):
        print(f"❌ 파일을 찾을 수 없습니다. 경로를 다시 확인해주세요: {image_path}")
        return
        
    print(f"🔍 이미지 분석을 시작합니다...")
    
    # 1. 이미지 로드 및 RGB 변환 (PNG 등에서 알파 채널 제거)
    raw_image = Image.open(image_path).convert("RGB")
    
    # 분석할 원본 이미지 화면에 출력
    display(raw_image.resize((256, 256)))
    
    # 2. 전처리 (이전 셀에서 정의한 preprocess 사용)
    img_tensor = preprocess(raw_image).unsqueeze(0).to(device)
    
    # 3. RDD 점수 계산
    score_rdd, s_img, s_lat = get_RDD_score(img_tensor)
    
    # 4. 결과 판별 및 리포트 출력
    print("\n" + "=" * 50)
    print(" 📊 [분석 결과 리포트]")
    print("=" * 50)
    print(f"🔹 S_image 점수 : {s_img:.5f}")
    print(f"🔹 S_latent 점수: {s_lat:.5f}")
    print(f"🏆 최종 RDD 점수: {score_rdd:.7f} (기준값: {threshold})")
    print("-" * 50)
    
    # 기준값(Threshold)과 비교하여 최종 판정
    if score_rdd > threshold:
        print("📸 최종 판정: 이 사진은 [원본(Real) 사진]일 확률이 높습니다!")
    else:
        print("🤖 최종 판정: 이 사진은 [AI가 생성한 이미지]일 확률이 높습니다!")
    print("=" * 50)

# =====================================================================
# 🚀 실행 영역 (여기에 복사한 파일 경로를 붙여넣으세요!)
# =====================================================================
my_file_path = "path/to/image.jpg"

# 만약 캐글 우측 패널 업로드가 번거롭다면, 
# 코드 셀에서 직접 다운로드 받은 임시 파일 경로를 사용해도 됩니다.
# 예: my_file_path = "/kaggle/working/test_image.jpg"

# classify_my_image(my_file_path, threshold=0.0001)


## 생성 이미지 20장 탐색 실험

In [ ]:
import random
import numpy as np
import torch
from tqdm.auto import tqdm

def evaluate_ai_generation_only(num_images=20, threshold=0.0001):
    """
    Stable Diffusion으로 이미지를 실시간 생성하고, 
    프레임워크가 이를 AI 이미지로 정확히 적발해내는지 테스트합니다.
    """
    print(f"🎨 Stable Diffusion으로 가짜(Fake) 이미지 {num_images}장을 생성하여 탐지력을 검증합니다...\n")
    
    # 다양성을 확보하기 위한 기본 프롬프트 풀
    sample_prompts = [
        "A hyper-realistic landscape photo of mountains at sunset",
        "A detailed portrait photography of a person reading a book in a cafe",
        "A macro shot of a morning dew drop on a green leaf",
        "A bustling city street at night with neon lights, 8k resolution",
        "A highly detailed photo of a delicious plate of pasta on a wooden table",
        "A cute golden retriever playing in a sunny park, cinematic lighting",
        "An old wooden cabin in the middle of a snowy forest, 4k photorealistic"
    ]

    # 시드 고정으로 실험 재현성 확보
    generator = torch.Generator(device).manual_seed(42)
    
    scores = []
    caught_count = 0

    for i in tqdm(range(num_images), desc="AI 이미지 생성 & 분석 중"):
        # 프롬프트를 무작위로 하나 골라서 이미지 생성 (속도를 위해 step=20)
        prompt = random.choice(sample_prompts)
        ai_image = pipe(prompt, num_inference_steps=20, generator=generator).images[0]
        
        # 생성된 이미지를 바로 전처리 후 RDD 점수 계산
        img_tensor = preprocess(ai_image).unsqueeze(0).to(device)
        score_rdd, s_img, s_lat = get_RDD_score(img_tensor)
        
        score_val = score_rdd.item() if hasattr(score_rdd, 'item') else score_rdd
        scores.append(score_val)
        
        # 임계값 이하인 경우 AI로 정확히 판별(적발)한 것으로 간주
        if score_val <= threshold:
            caught_count += 1
            
    # 통계 계산
    detection_rate = (caught_count / num_images) * 100
    avg_score = np.mean(scores)
    max_score = np.max(scores)
    min_score = np.min(scores)

    # ---------------------------------------------------------
    # 최종 리포트 출력
    # ---------------------------------------------------------
    print("\n" + "="*55)
    print(" 🛡️ [AI 생성 이미지 적발 (Fake Detection) 리포트]")
    print("="*55)
    print(f"🔸 적용된 임계값(Threshold) : {threshold}")
    print(f"🔸 총 생성된 AI 이미지 수   : {num_images}장")
    print("-" * 55)
    print(f"🚨 성공적으로 적발한 횟수   : {caught_count}장")
    print(f"🎯 최종 AI 이미지 적발률    : {detection_rate:.1f}%")
    print("-" * 55)
    print(" 📊 [RDD 점수 통계]")
    print(f"   - 평균 점수 : {avg_score:.7f}")
    print(f"   - 최고 점수 : {max_score:.7f} (가장 진짜 같았던 이미지)")
    print(f"   - 최저 점수 : {min_score:.7f} (가장 가짜 티가 났던 이미지)")
    print("="*55)

# =====================================================================
# 🚀 실행 영역
# =====================================================================
# 20장을 생성하고 임계값 0.0001 기준으로 방어력 테스트
evaluate_ai_generation_only(num_images=20, threshold=0.0001)

## Picsum 20장 탐색 실험

In [ ]:
import requests
import numpy as np
from PIL import Image
from io import BytesIO
from tqdm.auto import tqdm

def evaluate_internet_real_images(num_images=20, threshold=0.0001):
    """
    인터넷 오픈소스 API에서 실사 이미지를 실시간으로 다운로드하여
    프레임워크가 원본(Real)으로 올바르게 판별하는지 테스트합니다.
    """
    print(f"🌐 인터넷(Lorem Picsum)에서 {num_images}장의 무작위 실사 이미지를 수집하고 분석합니다...\n")
    
    scores = []
    correct_count = 0
    failed_downloads = 0

    # 다운로드 및 분석 루프
    for i in tqdm(range(num_images), desc="인터넷 Real 이미지 수집 & 분석 중"):
        try:
            # 고유한 이미지를 받기 위해 URL에 인덱스(i)를 시드로 활용
            url = f"https://picsum.photos/seed/{1007 + i}/512"
            
            # API 요청 및 이미지 로드 (메모리 상에서 바로 처리)
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            raw_image = Image.open(BytesIO(response.content)).convert("RGB")
            
            # 전처리 및 RDD 점수 계산 (이전 셀의 preprocess, get_RDD_score 활용)
            img_tensor = preprocess(raw_image).unsqueeze(0).to(device)
            score_rdd, s_img, s_lat = get_RDD_score(img_tensor)
            
            score_val = score_rdd.item() if hasattr(score_rdd, 'item') else score_rdd
            scores.append(score_val)
            
            # 임계값보다 점수가 크면 'Real'로 정확히 판별한 것으로 간주
            if score_val > threshold:
                correct_count += 1
                
        except Exception as e:
            # 일시적인 네트워크 오류 등으로 인한 실패 처리
            failed_downloads += 1
            print(f"\n⚠️ 이미지 다운로드 또는 처리 실패 (인덱스 {i}): {e}")

    # 유효한 데이터가 있는지 확인
    valid_count = num_images - failed_downloads
    if valid_count == 0:
        print("❌ 유효하게 분석된 이미지가 없습니다. 인터넷 연결 상태를 확인해 주세요.")
        return

    # 통계 계산
    pass_rate = (correct_count / valid_count) * 100
    avg_score = np.mean(scores)
    max_score = np.max(scores)
    min_score = np.min(scores)

    # ---------------------------------------------------------
    # 최종 리포트 출력
    # ---------------------------------------------------------
    print("\n" + "="*55)
    print(" 🌍 [오픈소스 API 실사(Real) 이미지 판별 리포트]")
    print("="*55)
    print(f"🔸 적용된 임계값(Threshold) : {threshold}")
    print(f"🔸 성공적으로 분석된 이미지 : {valid_count}장 (실패: {failed_downloads}장)")
    print("-" * 55)
    print(f"🛡️ 진짜 사진으로 통과시킨 횟수: {correct_count}장")
    print(f"🎯 최종 원본(Real) 정답률     : {pass_rate:.1f}%")
    print("-" * 55)
    print(" 📊 [RDD 점수 통계]")
    print(f"   - 평균 점수 : {avg_score:.7f}")
    print(f"   - 최고 점수 : {max_score:.7f}")
    print(f"   - 최저 점수 : {min_score:.7f}")
    print("="*55)

# =====================================================================
# 🚀 실행 영역
# =====================================================================
# 20장을 수집하고 임계값 0.0001 기준으로 테스트
evaluate_internet_real_images(num_images=20, threshold=0.0001)